In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, StandardScaler
import seaborn as sns
import sys


In [ ]:
df = pd.read_csv(sys.argv[1])

In [5]:
df.head()

,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-10-01 00:00:00 UTC,view,44600062,2103807459595387724,NaN,shiseido,35.79,541312140,72d76fde-8bb3-4e00-8c23-a032dfed738c
1,2019-10-01 00:00:00 UTC,view,3900821,2053013552326770905,appliances.environment.water_heater,aqua,33.20,554748717,9333dfbd-b87a-4708-9857-6336556b0fcc
2,2019-10-01 00:00:01 UTC,view,17200506,2053013559792632471,furniture.living_room.sofa,NaN,543.10,519107250,566511c2-e2e3-422b-b695-cf8e6e792ca8
3,2019-10-01 00:00:01 UTC,view,1307067,2053013558920217191,computers.notebook,lenovo,251.74,550050854,7c90fc70-0e80-4590-96f3-13c02c18c713
4,2019-10-01 00:00:04 UTC,view,1004237,2053013555631882655,electronics.smartphone,apple,1081.98,535871217,c6bd7419-2748-4c56-95b4-8cec9ff8b80d


In [6]:
df.columns

Index(['event_time', 'event_type', 'product_id', 'category_id',
       'category_code', 'brand', 'price', 'user_id', 'user_session'],
      dtype='object')

In [7]:
df.shape

(42448764, 9)

## data cleaning

In [8]:
df.duplicated().sum()

np.int64(30220)

In [9]:
df.isna().sum()

event_time              0
event_type              0
product_id              0
category_id             0
category_code    13515609
brand             6117080
price                   0
user_id                 0
user_session            2
dtype: int64

In [10]:
df['event_type'].unique()

array(['view', 'purchase', 'cart'], dtype=object)

In [11]:
df.drop(['event_time','product_id','category_id','user_id','user_session'],axis=1,inplace=True)

In [12]:
df.drop_duplicates(inplace=True)

In [13]:
df.dropna(inplace=True)

In [14]:
df.isna().sum()

event_type       0
category_code    0
brand            0
price            0
dtype: int64

In [15]:
df['category_code'].unique()

array(['appliances.environment.water_heater', 'computers.notebook',
       'electronics.smartphone', 'computers.desktop',
       'apparel.shoes.keds', 'appliances.kitchen.microwave',
       'furniture.bedroom.bed', 'electronics.video.tv',
       'appliances.kitchen.mixer', 'electronics.audio.headphone',
       'appliances.environment.air_heater', 'apparel.shoes',
       'appliances.environment.vacuum',
       'appliances.kitchen.refrigerators', 'appliances.kitchen.washer',
       'computers.peripherals.monitor', 'construction.tools.pump',
       'electronics.clocks', 'apparel.shoes.slipons',
       'appliances.kitchen.meat_grinder',
       'computers.components.videocards', 'construction.tools.drill',
       'kids.toys', 'electronics.telephone', 'furniture.bathroom.toilet',
       'auto.accessories.alarm', 'auto.accessories.player',
       'appliances.kitchen.grill', 'electronics.tablet',
       'appliances.kitchen.dishwasher', 'appliances.personal.hair_cutter',
       'kids.skates', '

In [16]:
df['category']=df['category_code'].str.split('.').str[0]

In [17]:
df['category'].value_counts()

category
electronics     60495
appliances      47088
computers       26376
construction    15100
kids             7956
furniture        6170
auto             3961
apparel          2846
accessories      2428
sport            1724
stationery        590
country_yard      424
medicine          214
Name: count, dtype: int64

In [18]:
df.drop('category_code', axis=1, inplace=True)

In [19]:
df['event_type'].unique()

array(['view', 'purchase', 'cart'], dtype=object)

In [20]:
df['price'].describe()

count    175372.000000
mean        331.311548
std         407.503203
min           0.880000
25%          71.170000
50%         175.400000
75%         419.552500
max        2574.070000
Name: price, dtype: float64

## Feature transformation

In [21]:
top_brands = df['brand'].value_counts().nlargest(10).index


In [22]:
df['brand'] = df['brand'].apply(lambda x: x if x in top_brands else 'other') 

In [23]:
df.head()

,event_type,brand,price,category
1,view,other,33.20,appliances
3,view,other,251.74,computers
4,view,apple,1081.98,electronics
5,view,other,908.62,computers
8,view,other,102.71,apparel


In [24]:

encode=LabelEncoder()

df['event_type_encoded']=encode.fit_transform(df['event_type'])

In [25]:
df['brand_type_encoded']=encode.fit_transform(df['brand'])

In [26]:
df['category_type_encoded']=encode.fit_transform(df['category'])

In [27]:
scaler = StandardScaler()
df['price_scaled'] = scaler.fit_transform(df[['price']])

In [28]:
from sklearn.decomposition import PCA


features = ['event_type_encoded', 'category_type_encoded', 'brand_type_encoded', 'price_scaled']


pca = PCA(n_components=3)  
pcs = pca.fit_transform(df[features])

df[['PC1', 'PC2', 'PC3']] = pcs

In [29]:
df.drop(['event_type_encoded',"brand_type_encoded",'category_type_encoded','price_scaled'],axis=1,inplace=True)

In [30]:
df.head()

,event_type,brand,price,category,PC1,PC2,PC3
1,view,other,33.20,appliances,2.663073,-1.623450,-0.482633
3,view,other,251.74,computers,1.301348,-0.139503,-0.002507
4,view,apple,1081.98,electronics,-6.950156,-3.096393,0.490947
5,view,other,908.62,computers,1.053524,-0.296644,1.582302
8,view,other,102.71,apparel,3.276487,-2.408192,-0.291365


In [31]:
print(df['price'].min(), df['price'].max())

0.88 2574.07


In [33]:
bins = [0, 50, 500, 1000, 2000, float('inf')] 
labels = ['budget', 'mid-range', 'premium', 'high-end', 'luxury']

df['price_bin'] = pd.cut(df['price'], bins=bins, labels=labels)

In [34]:
print(df.head())

  event_type  brand    price     category       PC1       PC2       PC3  \
1       view  other    33.20   appliances  2.663073 -1.623450 -0.482633   
3       view  other   251.74    computers  1.301348 -0.139503 -0.002507   
4       view  apple  1081.98  electronics -6.950156 -3.096393  0.490947   
5       view  other   908.62    computers  1.053524 -0.296644  1.582302   
8       view  other   102.71      apparel  3.276487 -2.408192 -0.291365   

   price_bin  
1     budget  
3  mid-range  
4   high-end  
5    premium  
8  mid-range  


In [ ]:
import subprocess
subprocess.run(["python", "analytics.py", "data_preprocessed.csv"])